# Pizzeria Amalfi: tuning the liquidity forecast (Optuna + MLflow)

`notebooks/forecast.ipynb` trains the forecast. This notebook **searches for better XGBoost settings** and makes the winner
promotable to production. It is a thin front over `cfo.forecast.tune`, so the same search also runs headless:
`python -m cfo.forecast.tune --trials 200 --promote-if-better`.

What it does, top to bottom:

1. load the bank history (Postgres when `DATABASE_URL` works, else the organizer's CSV),
2. score today's model with the **rolling-origin backtest** (`cfo.forecast.evaluate`): at month ends the model is retrained on
   the past only and its predicted **balance** is compared with what happened over the next 90 days,
3. run an **Optuna** search (TPE, seeded) over the feature set and the XGBoost parameters; every trial is a child run of one
   MLflow parent run `tuning-<date>` in the experiment `liquidity-forecast-tuning`,
4. plot the optimisation history, the parameter importance, and predicted vs real balance of the best and the current model,
5. show the top-10 trials (their spread tells how much of a gain is noise),
6. **promote** the winner: retrain on the full history, register it in the MLflow model registry as `challenger`
   (and, only if `PROMOTE_IF_BETTER` is on and it beats the current champion by the margin, as `liquidity-forecast@champion`).

Needs only `MLFLOW_TRACKING_URI` (optionally `DATABASE_URL`, `SOURCE_CSV`). Without a tracking URI the search still runs, it just
logs nothing and cannot promote. Only aggregates are shown, never raw statement rows.

In [ ]:
# Run once per workbench, then restart the kernel. Uncomment the lines you need.
# %pip install -e "backend[mlflow,tuning]"   # the cfo library with mlflow, optuna and matplotlib

import os

# os.chdir(os.path.expanduser("~/house"))                        # only if the kernel does not start inside the repo
# os.environ["MLFLOW_TRACKING_URI"] = "http://mlflow:5000"      # or sqlite:////tmp/mlflow.db for a local try-out
# os.environ["DATABASE_URL"] = "postgresql://cfo:<password>@postgres:5432/cfo"   # optional, else the CSV

N_TRIALS = 200               # the one knob: 200 trials take a few minutes on 2 CPUs
SEED = 0                     # same seed, same data -> same best parameters
MIN_GAIN = 0.02              # a candidate is "better" only by 2 % of the reference's balance error, not by noise
PROMOTE_IF_BETTER = False    # True: move liquidity-forecast@champion when the winner beats it by MIN_GAIN

## 1. Imports

In [ ]:
import dataclasses
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from cfo.forecast import evaluate as ev
from cfo.forecast import features as feat
from cfo.forecast import model as fm
from cfo.forecast import tune  # the logic of this notebook: search, promote, load_transactions

BLUE, ORANGE, GREY, RED, GREEN = "#1f77b4", "#e07b00", "#7f8c9a", "#c0392b", "#2e8b57"
pd.options.display.float_format = "{:,.1f}".format
print("MLflow tracking:", os.environ.get("MLFLOW_TRACKING_URI") or "not set (no logging, no promotion)")

## 2. Load the transactions

`tune.load_transactions()` tries the company Postgres first and falls back to the bank statement CSV.

**December is held out.** Everything below (baseline, search, plots of the rolling folds) uses the bookings **up to 30 Nov 2025**
only. December 2025 is never seen by the search; it is used once, in section 6, as an independent check of the winner. The
promotion in section 8 then retrains on the full history.

In [ ]:
tx_all, SOURCE = tune.load_transactions("auto")
HISTORY_END = pd.Timestamp("2025-11-30")          # the search sees nothing after this day
tx = tx_all[tx_all["booking_date"] <= HISTORY_END].reset_index(drop=True)
print(f"{len(tx_all)} bookings from {SOURCE}: {tx_all['booking_date'].min().date()} .. {tx_all['booking_date'].max().date()}")
print(f"search history: {len(tx)} bookings up to {HISTORY_END.date()}; "
      f"held out: {len(tx_all) - len(tx)} bookings after it")

## 3. The baseline: today's model on the rolling folds

`DEFAULT_FEATURE_SET` with its current parameters. The objective of the search is `balance_error_90d`, the mean absolute error of the
predicted balance 90 days after the origin. The mean is dominated by the 30 Jun origin (the revenue step of July cannot be known
from January to June), so the per-origin errors below are the more honest picture.

In [ ]:
baseline_set, baseline_params = fm.DEFAULT_FEATURE_SET, tune.baseline_params()
base_folds = ev.rolling_origin(tx, tune.config_trainer(baseline_set, baseline_params))
base_scores = ev.score(base_folds)
print(f"baseline: feature set {baseline_set}, params {json.dumps(baseline_params)}")
display(pd.Series(tune.finite(base_scores), name="baseline").to_frame())
display(pd.DataFrame(tune.per_origin_errors(base_folds)).T.rename_axis("origin"))

## 4. The search

`tune.search` draws a feature set (`calendar`, `level`, `level_baseline`) and the parameters `n_estimators`, `max_depth`,
`learning_rate`, `min_child_weight`, `subsample`, `colsample_bytree`, `reg_lambda`, `reg_alpha`, `gamma` and the loss
(`reg:squarederror` or `reg:absoluteerror`) within small-model ranges. The current defaults are trial 0, so the result is never worse
than not tuning. Each trial trains about six small models; progress and the MLflow link are in the output.

In [ ]:
result = tune.search(tx, n_trials=N_TRIALS, seed=SEED, min_gain=MIN_GAIN, data_source=SOURCE)
key = result.objective_metric
print(f"{result.n_trials} trials, {result.seconds_per_trial:.1f} s per trial")
print(f"{key}: baseline {result.baseline_scores[key]:,.0f} -> best {result.best_scores[key]:,.0f} "
      f"(better by the {MIN_GAIN:.0%} margin: {result.beats_baseline()})")
ex = tune.error_without_first_origin
print(f"mean |error at the end of the fold| without the first origin: baseline {ex(result.baseline_per_origin):,.0f} "
      f"-> best {ex(result.per_origin):,.0f}")
print("best feature set:", result.best_feature_set)
print("best params:", json.dumps(result.best_params))
if result.parent_run_id:
    print("MLflow parent run:", result.parent_run_id, "in experiment", result.experiment)

## 5. Optimisation history and parameter importance

Left: the objective of every trial and the best so far against today's defaults. Right: how much each parameter explains of the
objective (Optuna's fANOVA); with a few hundred trials this is a hint, not a proof.

In [ ]:
trials = pd.DataFrame([t for t in result.trials if not t.get("error")])
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4), gridspec_kw={"width_ratios": [3, 2]})
ax1.scatter(trials["number"], trials["objective"], s=14, color=GREY, label="trial")
ax1.plot(trials["number"], trials["objective"].cummin(), color=BLUE, lw=2, label="best so far")
ax1.axhline(result.baseline_scores[key], color=ORANGE, ls="--", lw=1, label="current defaults")
ax1.set_xlabel("trial"); ax1.set_ylabel(f"{key} (EUR)"); ax1.set_title("Optimisation history"); ax1.legend()
try:
    importance = tune.param_importance(result)
    ax2.barh(importance.index[::-1], importance.values[::-1], color=BLUE)
    ax2.set_title("Parameter importance")
except Exception as exc:  # noqa: BLE001 - too few trials for the importance estimate
    ax2.text(0.5, 0.5, f"not available: {type(exc).__name__}", ha="center"); ax2.set_axis_off()
fig.tight_layout(); plt.show()

## 6. Predicted vs real balance: best vs baseline

For every origin the model is retrained on the past and the **cumulative** predicted net flow is compared with the real one (the
balance error). A good model's dashed line follows the solid one.

In [ ]:
best_folds = ev.rolling_origin(tx, tune.config_trainer(result.best_feature_set, result.best_params))
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, folds, title in ((axes[0], base_folds, f"baseline ({baseline_set})"),
                         (axes[1], best_folds, f"best ({result.best_feature_set})")):
    for origin, fold in folds.groupby("origin"):
        pred, real = fold["predicted_net"].cumsum(), fold["actual_net"].cumsum()
        ax.plot(fold["date"], real, color=GREY, lw=1.5)
        ax.plot(fold["date"], pred, color=BLUE, lw=1.2, ls="--")
    ax.set_title(f"{title}: cumulative net, real (grey) vs predicted (blue)")
    ax.tick_params(axis="x", rotation=30)
axes[0].set_ylabel("EUR since the origin")
fig.tight_layout(); plt.show()

per_origin = pd.concat({"baseline": pd.DataFrame(result.baseline_per_origin).T,
                        "best": pd.DataFrame(result.per_origin).T}, axis=1)
display(per_origin)

### December 2025: the held-out check

The search never saw December. Both models are trained on the bookings up to 30 Nov 2025 (the search history) and forecast December,
which is then compared with what was booked. This is the only out-of-sample test of the winner in this notebook; it needs a history
that reaches into December (the CSV does, the Postgres history may not).

In [ ]:
dec = pd.date_range("2025-12-01", "2025-12-31")
if tx_all["booking_date"].max() < dec[-1]:
    print("The history ends before 31 Dec 2025: no held-out December comparison.")
else:
    past = tx  # the search history, up to 30 Nov
    truth = feat.daily_cash_flow(tx_all, start=dec[0], end=dec[-1], exclude_categories=fm.EXCLUDED_CATEGORIES)["net_cash_flow"]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(dec, truth.cumsum(), color=GREY, lw=2, label="real")
    for label, fs, params, color in (("baseline", baseline_set, baseline_params, ORANGE),
                                     ("best", result.best_feature_set, result.best_params, BLUE)):
        model = fm.train(past, feature_set=fs, params=params, log_to_mlflow=False, holdout_days=0)
        pred = model.predict(dec)
        print(f"{label}: December daily MAE {np.abs(pred.to_numpy() - truth.to_numpy()).mean():,.0f} EUR, "
              f"balance error after 31 days {pred.sum() - truth.sum():+,.0f} EUR")
        ax.plot(dec, pred.cumsum(), color=color, lw=1.5, ls="--", label=label)
    ax.set_title("December 2025 (held out): cumulative net flow (EUR)"); ax.legend(); fig.tight_layout(); plt.show()

## 7. The top-10 trials

The best ten trials and their spread. If the ten are within a few percent of each other the optimum is flat and any of them will do; if
the best stands alone, suspect a lucky fit to four or five folds. A candidate counts as better than the baseline only by the
`MIN_GAIN` margin.

In [ ]:
display(result.top(10))
spread = result.top10_spread()
print({k: round(v, 1) for k, v in spread.items()})
print("beats the baseline by the margin:", result.beats_baseline())

## 8. Promote the winner

Retrains the best configuration on the **full** history (including December, which was only held out for the check above), logs it through the existing MLflow path and registers a new version of
`liquidity-forecast` with the alias `challenger`. With `PROMOTE_IF_BETTER = True` the alias `champion` (what the API serves when
`FORECAST_MODEL_URI` points at it) moves only if the new model beats the current champion's `balance_error_90d` by `MIN_GAIN`,
both scored now on this data. Otherwise it stays `challenger` and the reason is printed.

In [ ]:
if os.environ.get("MLFLOW_TRACKING_URI"):
    full = dataclasses.replace(result, transactions=tx_all)   # retrain (and score the champion) on all bookings
    promotion = tune.promote(full, if_better=PROMOTE_IF_BETTER, min_gain=MIN_GAIN)
    print(f"version {promotion.version} registered as {promotion.alias}; champion moved: {promotion.moved_champion}")
    print(promotion.reason)
else:
    print("MLFLOW_TRACKING_URI is not set: nothing to promote to.")